In [1]:
# Airbnb Price Prediction - End-to-End Machine Learning Project

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the dataset

df = pd.read_csv("../data/airbnb.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (48900, 16)


,id,name,host_id,host_name,neighbourhood group,neighbourhood,latitude,longitude,room type,price,minimum nights,number_of_reviews,last review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149.0,1.0,9,10/19/2018,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225.0,1.0,45,5/21/2019,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150.0,3.0,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89.0,1.0,270,7/5/2019,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80.0,10.0,9,11/19/2018,0.10,1,0


In [4]:
# Inspect dataset structure and data types

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48900 entries, 0 to 48899
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48900 non-null  int64  
 1   name                            48884 non-null  str    
 2   host_id                         48900 non-null  int64  
 3   host_name                       48879 non-null  str    
 4   neighbourhood group             48900 non-null  str    
 5   neighbourhood                   48900 non-null  str    
 6   latitude                        48900 non-null  float64
 7   longitude                       48900 non-null  float64
 8   room type                       48900 non-null  str    
 9   price                           48900 non-null  float64
 10  minimum nights                  48900 non-null  float64
 11  number_of_reviews               48900 non-null  int64  
 12  last review                     38848 non-n

In [5]:
# Check missing values

missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values[missing_values > 0])

Missing values in each column:
name                    16
host_name               21
last review          10052
reviews_per_month    10052
dtype: int64


In [7]:
# Clean the dataset

# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing review counts with 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Drop columns that are not useful for predicting price
columns_to_drop = [
    "id",
    "name",
    "host_id",
    "host_name",
    "last review"
]

df = df.drop(columns=columns_to_drop)

print("Data cleaning completed!")
print("New shape:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Data cleaning completed!
New shape: (48895, 11)

Remaining missing values:
Series([], dtype: int64)


In [8]:
# Separate features and target

X = df.drop(columns=["price"])
y = df["price"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

Features (X) shape: (48895, 10)
Target (y) shape: (48895,)

Feature columns:
['neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'room type', 'minimum nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']


In [9]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (39116, 10)
Testing features: (9779, 10)
Training target: (39116,)
Testing target: (9779,)


In [11]:
# Identify numerical and categorical features

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['latitude', 'longitude', 'minimum nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

Categorical features:
['neighbourhood group', 'neighbourhood', 'room type']


In [12]:
# Create preprocessing pipeline

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [21]:
# Create the complete machine learning pipeline

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

print("Machine learning pipeline created successfully!")

Machine learning pipeline created successfully!


In [22]:
# Train the machine learning model

print("Training model...")

model_pipeline.fit(X_train, y_train)

print("Model training completed successfully!")

Training model...
Model training completed successfully!


In [23]:
# Generate predictions on the test data

y_pred = model_pipeline.predict(X_test)

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

Predictions generated successfully!
Number of predictions: 9779


In [24]:
# Evaluate model performance

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Results")
print("-" * 30)
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R² Score: {r2:.4f}")

Model Evaluation Results
------------------------------
Mean Squared Error (MSE): 38176.43
Root Mean Squared Error (RMSE): 195.39
Mean Absolute Error (MAE): 68.07
R² Score: 0.1370


In [25]:
# Save the trained model

model_path = "../model/airbnb_price_model.pkl"

joblib.dump(model_pipeline, model_path)

print("Model saved successfully!")
print(f"Saved to: {model_path}")

Model saved successfully!
Saved to: ../model/airbnb_price_model.pkl


In [26]:
# Verify that the saved model can be loaded

loaded_model = joblib.load(model_path)

print("Saved model loaded successfully!")
print("Model type:", type(loaded_model).__name__)

Saved model loaded successfully!
Model type: Pipeline


In [27]:
# Test the loaded model

loaded_predictions = loaded_model.predict(X_test)

print("Loaded model prediction test successful!")
print("Number of predictions:", len(loaded_predictions))

Loaded model prediction test successful!
Number of predictions: 9779
